<a href="https://colab.research.google.com/github/ehas1/Statistical-Bias-in-ML/blob/main/LIME_and_SHAP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Import required libraries
%%capture
! pip install lime
import lime.lime_tabular
import shap
import numpy as np
import seaborn as sns
import os
import glob
import joblib
import matplotlib.pyplot as plt
import pandas as pd
import requests
from pathlib import Path
import xgboost as xgb
from joblib import dump, load
from io import StringIO

# Load and preprocess data
def load_data(url="https://raw.githubusercontent.com/propublica/compas-analysis/refs/heads/master/cox-violent-parsed.csv"):
    response = requests.get(url)
    data = pd.read_csv(StringIO(response.text))
    return data

# We need this to ensure we're using the same preprocessing as the training
from OAIP_Skeleton import preprocess_data

In [ ]:
# Model Loading
def load_latest_models(models_dir="/content/drive/MyDrive/COMPAS_models"):
    """Load the latest version of each model from cloud storage."""
    try:
        # Find latest model files
        dt_files = list(Path(models_dir).glob('decision_tree_*.joblib'))
        xgb_files = list(Path(models_dir).glob('xgboost_*.json'))
        nn_files = list(Path(models_dir).glob('neural_network_*.keras'))

        if not (dt_files and xgb_files and nn_files):
            raise FileNotFoundError("Could not find all required model files")

        # Get latest version of each model
        latest_dt = max(dt_files, key=lambda p: p.stat().st_ctime)
        latest_xgb = max(xgb_files, key=lambda p: p.stat().st_ctime)
        latest_nn = max(nn_files, key=lambda p: p.stat().st_ctime)

        print(f"Loading models:\n{latest_dt}\n{latest_xgb}\n{latest_nn}")

        # Load models
        dt_model = joblib.load(latest_dt)
        xgb_model = xgb.XGBClassifier()
        xgb_model.load_model(str(latest_xgb))
        nn_model = load_model(latest_nn)

        return dt_model, xgb_model, nn_model

    except Exception as e:
        print(f"Error loading models: {str(e)}")
        print("Please ensure you have run OAIP_Skeleton.ipynb first to save the models.")
        raise

# Load data and preprocess it the same way as during training
print("Loading and preprocessing data...")
data = load_data()
train_data, test_data = preprocess_data(data)
X_train, y_train = train_data
X_test, y_test = test_data

# Load the models
print("\nLoading models...")
dt_model, xgb_model, nn_model = load_latest_models()
print("Successfully loaded all models!")

# LIME vs. SHAP

In [ ]:
# LIME and SHAP Analysis

# Find the most recent model files
def find_latest_model(pattern):
    # Get the models directory path
    models_dir = os.path.join(os.path.expanduser('~'), 'Desktop', 'COMPAS_models')
    # Search for files in the models directory
    files = glob.glob(os.path.join(models_dir, pattern))
    if not files:
        raise FileNotFoundError(f"No model files found matching pattern: {pattern}")
    return max(files, key=os.path.getctime)

try:
    # Find and load the most recent models
    dt_model_path = find_latest_model('model_*.joblib')  # Decision tree uses joblib
    xgb_model_path = find_latest_model('model_*.json')   # XGBoost uses json
    nn_model_path = find_latest_model('model_*.keras')   # Neural network uses keras

    print(f"Loading models from:\n{dt_model_path}\n{xgb_model_path}\n{nn_model_path}")

    # Load models using appropriate methods
    dt_model = load(dt_model_path)
    xgb_model = xgb.XGBClassifier()
    xgb_model.load_model(xgb_model_path)
    nn_model = load_model(nn_model_path)

except FileNotFoundError as e:
    print("Error: Could not find saved model files.")
    print("Please ensure you have run the Decision Tree, XGBoost, and Neural Network cells first.")
    print("The models should be saved in the current working directory.")
    print(f"\nSpecific error: {str(e)}")
    raise

# Set up the explainers
def setup_explainers(train_data, feature_names):
    """Set up LIME and SHAP explainers for all models."""
    X_train, _ = train_data

    # LIME explainer
    lime_explainer = lime.lime_tabular.LimeTabularExplainer(
        X_train.values,
        feature_names=feature_names,
        class_names=['No Recidivism', 'Recidivism'],
        mode='classification'
    )

    # SHAP explainers
    dt_explainer = shap.TreeExplainer(dt_model)
    xgb_explainer = shap.TreeExplainer(xgb_model)

    # For neural network, use KernelExplainer with a background dataset
    background = shap.sample(X_train, 100)  # Use 100 background samples
    nn_predict = lambda x: nn_model.predict(x)
    nn_explainer = shap.KernelExplainer(nn_predict, background)

    return lime_explainer, (dt_explainer, xgb_explainer, nn_explainer)

# Generate explanations
def explain_instance(instance, instance_idx, lime_exp, shap_exp, feature_names):
    """Generate and plot LIME and SHAP explanations for all models."""
    dt_exp, xgb_exp, nn_exp = shap_exp

    # Set up the plot
    plt.figure(figsize=(15, 10))

    # LIME explanations
    plt.subplot(2, 3, 1)
    lime_dt = lime_exp.explain_instance(
        instance.values,
        dt_model.predict_proba,
        num_features=6
    )
    lime_dt.as_pyplot_figure()
    plt.title('LIME - Decision Tree')
    plt.tight_layout()

    plt.subplot(2, 3, 2)
    lime_xgb = lime_exp.explain_instance(
        instance.values,
        xgb_model.predict_proba,
        num_features=6
    )
    lime_xgb.as_pyplot_figure()
    plt.title('LIME - XGBoost')
    plt.tight_layout()

    plt.subplot(2, 3, 3)
    lime_nn = lime_exp.explain_instance(
        instance.values,
        nn_model.predict,
        num_features=6
    )
    lime_nn.as_pyplot_figure()
    plt.title('LIME - Neural Network')
    plt.tight_layout()

    # SHAP explanations
    plt.subplot(2, 3, 4)
    shap_values_dt = dt_exp.shap_values(instance.values.reshape(1, -1))
    shap.force_plot(
        dt_exp.expected_value[1] if isinstance(dt_exp.expected_value, list)
        else dt_exp.expected_value,
        shap_values_dt[1] if isinstance(shap_values_dt, list) else shap_values_dt,
        instance,
        feature_names=feature_names,
        matplotlib=True,
        show=False
    )
    plt.title('SHAP - Decision Tree')

    plt.subplot(2, 3, 5)
    shap_values_xgb = xgb_exp.shap_values(instance.values.reshape(1, -1))
    shap.force_plot(
        xgb_exp.expected_value,
        shap_values_xgb,
        instance,
        feature_names=feature_names,
        matplotlib=True,
        show=False
    )
    plt.title('SHAP - XGBoost')

    plt.subplot(2, 3, 6)
    shap_values_nn = nn_exp.shap_values(instance.values.reshape(1, -1))
    shap.force_plot(
        nn_exp.expected_value,
        shap_values_nn,
        instance,
        feature_names=feature_names,
        matplotlib=True,
        show=False
    )
    plt.title('SHAP - Neural Network')

    plt.tight_layout()
    plt.show()

# Select instances to explain
def get_interesting_instances(X_test, y_test, models, n_instances=3):
    """Select interesting instances where models disagree or are uncertain."""
    dt_model, xgb_model, nn_model = models

    dt_pred = dt_model.predict_proba(X_test)[:, 1]
    xgb_pred = xgb_model.predict_proba(X_test)[:, 1]
    nn_pred = nn_model.predict(X_test).ravel()

    # Calculate disagreement score
    mean_pred = (dt_pred + xgb_pred + nn_pred) / 3
    disagreement = np.std([dt_pred, xgb_pred, nn_pred], axis=0)

    # Find instances with high disagreement and different true labels
    interesting_indices = []
    seen_labels = set()

    # Sort by disagreement
    sorted_indices = np.argsort(disagreement)[::-1]

    for idx in sorted_indices:
        true_label = y_test.iloc[idx]
        if true_label not in seen_labels and len(interesting_indices) < n_instances:
            interesting_indices.append(idx)
            seen_labels.add(true_label)

    return interesting_indices

# Generate summary plots
def generate_summary_plots(X_train, shap_explainers):
    """Generate and display summary plots for all models."""
    dt_exp, xgb_exp, nn_exp = shap_explainers

    plt.figure(figsize=(15, 10))

    # Decision Tree summary plot
    plt.subplot(1, 3, 1)
    shap_values_dt = dt_exp.shap_values(X_train)
    if isinstance(shap_values_dt, list):
        shap.summary_plot(shap_values_dt[1], X_train, plot_size=(5, 5), show=False)
    else:
        shap.summary_plot(shap_values_dt, X_train, plot_size=(5, 5), show=False)
    plt.title('SHAP Summary - Decision Tree')

    # XGBoost summary plot
    plt.subplot(1, 3, 2)
    shap_values_xgb = xgb_exp.shap_values(X_train)
    shap.summary_plot(shap_values_xgb, X_train, plot_size=(5, 5), show=False)
    plt.title('SHAP Summary - XGBoost')

    # Neural Network summary plot
    plt.subplot(1, 3, 3)
    shap_values_nn = nn_exp.shap_values(shap.sample(X_train, 100))  # Sample for efficiency
    shap.summary_plot(shap_values_nn, shap.sample(X_train, 100), plot_size=(5, 5), show=False)
    plt.title('SHAP Summary - Neural Network')

    plt.tight_layout()
    plt.show()

# Main execution
feature_names = X_train.columns.tolist()

# Set up explainers
lime_explainer, shap_explainers = setup_explainers(train_data, feature_names)

# Generate summary plots
print("Generating SHAP Summary Plots...")
generate_summary_plots(X_train, shap_explainers)

# Get interesting instances
print("\nGenerating Individual Instance Explanations...")
interesting_indices = get_interesting_instances(X_test, y_test, (dt_model, xgb_model, nn_model))

# Generate explanations for interesting instances
for idx in interesting_indices:
    instance = X_test.iloc[idx]
    true_label = y_test.iloc[idx]
    print(f"\nExplaining instance {idx} (True label: {'Recidivism' if true_label == 1 else 'No Recidivism'})")
    print(f"Predictions: DT: {dt_model.predict_proba(instance.values.reshape(1, -1))[0, 1]:.3f}, "
          f"XGB: {xgb_model.predict_proba(instance.values.reshape(1, -1))[0, 1]:.3f}, "
          f"NN: {nn_model.predict(instance.values.reshape(1, -1))[0, 0]:.3f}")
    explain_instance(instance, idx, lime_explainer, shap_explainers, feature_names)
